# Xuất model fine-tune sang GGUF để chạy bằng Ollama

**Settings:** Accelerator = *None* (CPU), Internet = *On*,
Add Input = dataset chứa `best_adapter` (ví dụ `modelhuhuver1`).

Các bước: gộp LoRA vào trọng số Qwen2.5-7B-Instruct (bf16) → chuyển sang GGUF f16 bằng llama.cpp → lượng tử hóa Q4_K_M (~4.7 GB).
Chạy *Save & Run All*, xong thì tải `qwen2.5-7b-huhu-Q4_K_M.gguf` ở tab Output về cùng thư mục với `serving/Modelfile`.

Bước gộp làm trực tiếp trên file safetensors theo công thức của LoRA: `W = W + (B @ A) * lora_alpha / r`
(không dùng transformers / peft, để tránh xung đột phiên bản thư viện trên Kaggle).

In [ ]:
!df -h /tmp /kaggle/working | cat
!free -g | cat
!git clone --depth 1 https://github.com/ggml-org/llama.cpp

In [ ]:
import glob, json, os, shutil
import torch
from huggingface_hub import snapshot_download
from safetensors import safe_open
from safetensors.torch import load_file, save_file

BASE = "Qwen/Qwen2.5-7B-Instruct"
OUT = "/tmp/merged"
hits = glob.glob("/kaggle/input/**/best_adapter/adapter_model.safetensors", recursive=True)     or glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
assert hits, "Không thấy adapter_model.safetensors trong /kaggle/input - hãy Add Input dataset chứa best_adapter"
ADAPTER = os.path.dirname(hits[0])
cfg = json.load(open(os.path.join(ADAPTER, "adapter_config.json")))
assert not cfg.get("use_rslora") and not cfg.get("use_dora") and not cfg.get("fan_in_fan_out")
SCALE = cfg["lora_alpha"] / cfg["r"]

# Tên tensor gốc -> (khóa lora_A, khóa lora_B)
pairs = {}
with safe_open(os.path.join(ADAPTER, "adapter_model.safetensors"), "pt") as f:
    for k in f.keys():
        if k.endswith(".lora_A.weight"):
            pairs[k.replace("base_model.model.", "", 1).replace(".lora_A.weight", ".weight")] = (k, k.replace("lora_A", "lora_B"))
print(f"Adapter: {ADAPTER} | {len(pairs)} ma trận LoRA | scale = {SCALE}")

src = snapshot_download(BASE, allow_patterns=["*.json", "*.safetensors", "*.txt"])
os.makedirs(OUT, exist_ok=True)
merged = set()
with safe_open(os.path.join(ADAPTER, "adapter_model.safetensors"), "pt") as lora:
    for shard in sorted(glob.glob(os.path.join(src, "*.safetensors"))):
        t = load_file(shard)
        for name in t:
            if name in pairs:
                a, b = pairs[name]
                A, B = lora.get_tensor(a).float(), lora.get_tensor(b).float()
                t[name] = (t[name].float() + (B @ A) * SCALE).to(t[name].dtype)
                merged.add(name)
        save_file(t, os.path.join(OUT, os.path.basename(shard)), metadata={"format": "pt"})
        print("  xong", os.path.basename(shard))
        del t
assert merged == set(pairs), f"Thiếu: {sorted(set(pairs) - merged)[:5]}"
for fn in os.listdir(src):
    if not fn.endswith(".safetensors"):
        shutil.copy(os.path.join(src, fn), OUT)
print("Đã gộp", len(merged), "ma trận ->", sorted(os.listdir(OUT)))

In [ ]:
# Chỉ cài gói gguf của llama.cpp, không kéo theo phụ thuộc (giữ nguyên torch / transformers có sẵn)
%pip install -q --no-deps ./llama.cpp/gguf-py
%pip install -q sentencepiece
!python llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/huhu-f16.gguf --outtype f16
!ls -lh /tmp/huhu-f16.gguf

In [ ]:
!rm -rf /tmp/merged
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize /tmp/huhu-f16.gguf /kaggle/working/qwen2.5-7b-huhu-Q4_K_M.gguf Q4_K_M
!rm -f /tmp/huhu-f16.gguf
!ls -lh /kaggle/working